# Part 2: Vault Configuration
## Agentic Security Hands-On Lab

**Estimated Time:** 20 minutes

**Prerequisites:** Complete Part 1 (Introduction and Setup)

---

## 🎯 Learning Objectives

In this notebook, you will:

1. Configure HashiCorp Vault for JWT authentication
2. Set up the Database Secrets Engine
3. Create and understand Vault policies
4. Configure external group mapping to IBM Verify
5. Understand policy-based access control

---

## 📚 What is HashiCorp Vault?

HashiCorp Vault is a secrets management tool that provides:

- **Secrets Management** - Securely store and access secrets
- **Dynamic Secrets** - Generate credentials on-demand
- **Data Encryption** - Encrypt data in transit and at rest
- **Identity-Based Access** - Fine-grained access control
- **Audit Logging** - Complete audit trail of all operations

### Key Concepts:

1. **Auth Methods** - How agents authenticate to Vault (JWT, LDAP, etc.)
2. **Secrets Engines** - Generate or store secrets (Database, AWS, etc.)
3. **Policies** - Define what authenticated agents can do
4. **External Groups** - Map external identity provider groups to Vault policies

---

## 🔧 Setup Environment

In [3]:
import os
import sys
from pathlib import Path
import subprocess

# Add parent directory to path
# Jupyter notebooks run with cwd set to agentic-security-incubation
lab_root = Path.cwd().parent
sys.path.insert(0, str(lab_root))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(lab_root / '.env')

print(f"📁 Lab Root: {lab_root}")
print(f"🔐 Vault Address: {os.getenv('VAULT_ADDR', 'http://localhost:8200')}")

📁 Lab Root: /Users/stevendirjayanto/initiatives/agentic-security/agentic-security-incubation
🔐 Vault Address: http://localhost:8200


---

## 🚀 Configure Vault

We'll use Terraform to configure Vault. Terraform provides:

- **Declarative Configuration** - Define desired state, not steps
- **Idempotent** - Safe to run multiple times
- **State Management** - Tracks what's been created
- **Plan Before Apply** - Preview changes before making them

The Terraform configuration will:

1. Enable JWT authentication method
2. Configure JWT auth to trust IBM Verify
3. Enable Database Secrets Engine
4. Configure PostgreSQL connection
5. Create database roles (hr-basic-reader, hr-admin-reader)
6. Create Vault policies
7. Set up external group mapping

Let's run the Terraform configuration:

In [4]:
print("🔐 Configuring HashiCorp Vault with Terraform...\n")
print("This will set up:")
print("  • JWT Authentication")
print("  • Database Secrets Engine")
print("  • Access Policies")
print("  • External Group Mapping\n")

terraform_dir = lab_root / 'infrastructure' / 'vault' / 'terraform'

# Check if terraform.tfvars exists
tfvars_file = terraform_dir / 'terraform.tfvars'
if not tfvars_file.exists():
    print("⚠️  terraform.tfvars file not found!")
    print(f"\n📝 Please create the file at: {tfvars_file}")
    print("\nYou can copy from the example:")
    print(f"  cd {terraform_dir}")
    print("  cp terraform.tfvars.example terraform.tfvars")
    print("\nThen edit terraform.tfvars with your actual values:")
    print("  • vault_addr (default: http://localhost:8200)")
    print("  • vault_token (default: root)")
    print("  • postgres_* settings")
    print("  • ibm_verify_* settings")
    raise FileNotFoundError(f"terraform.tfvars not found at {tfvars_file}")
else:
    print(f"✅ Found terraform.tfvars at: {tfvars_file}\n")

# Change to terraform directory
os.chdir(terraform_dir)

# Initialize Terraform (if not already done)
print("📦 Initializing Terraform...")
init_result = subprocess.run(
    ['terraform', 'init'],
    capture_output=True,
    text=True
)

if init_result.returncode != 0:
    print("❌ Terraform init failed:")
    print(init_result.stderr)
else:
    print("✅ Terraform initialized\n")

# Apply Terraform configuration
print("🚀 Applying Terraform configuration...")
print("   Using configuration from terraform.tfvars file\n")
apply_result = subprocess.run(
    ['terraform', 'apply', '-auto-approve'],
    capture_output=True,
    text=True
)

if apply_result.returncode == 0:
    print("✅ Vault configuration successful!\n")
    print("=" * 80)
    print(apply_result.stdout)
    print("=" * 80)
    
    # Show outputs
    print("\n📊 Terraform Outputs:")
    output_result = subprocess.run(
        ['terraform', 'output', '-json'],
        capture_output=True,
        text=True
    )
    if output_result.returncode == 0:
        import json
        outputs = json.loads(output_result.stdout)
        for key, value in outputs.items():
            print(f"  • {key}: {value.get('value', 'N/A')}")
else:
    print("❌ Error configuring Vault:")
    print(apply_result.stderr)
    print("\nTroubleshooting:")
    print("  1. Ensure Vault container is running")
    print("  2. Check VAULT_ADDR and VAULT_TOKEN in .env")
    print("  3. Verify Vault is unsealed")
    print("  4. Ensure Terraform is installed (terraform --version)")

# Change back to lab root
os.chdir(lab_root)

print("\n📝 Configuration Summary:")
print("  ✅ JWT Authentication enabled")
print("  ✅ Database Secrets Engine configured")
print("  ✅ Policies created (hr-basic, hr-admin)")
print("  ✅ External groups mapped to IBM Verify")
print("  ✅ Database roles created (hr-basic-reader, hr-admin-reader)")

🔐 Configuring HashiCorp Vault with Terraform...

This will set up:
  • JWT Authentication
  • Database Secrets Engine
  • Access Policies
  • External Group Mapping

✅ Found terraform.tfvars at: /Users/stevendirjayanto/initiatives/agentic-security/agentic-security-incubation/infrastructure/vault/terraform/terraform.tfvars

📦 Initializing Terraform...
✅ Terraform initialized

🚀 Applying Terraform configuration...
   Using configuration from terraform.tfvars file

✅ Vault configuration successful!

vault_audit.file: Refreshing state... [id=file]
vault_policy.hr_basic: Refreshing state... [id=hr-basic-policy]
vault_policy.hr_admin: Refreshing state... [id=hr-admin-policy]
vault_jwt_auth_backend.ibm_verify: Refreshing state... [id=jwt]
vault_database_secrets_mount.db: Refreshing state... [id=database]
vault_identity_group.hr_basic: Refreshing state... [id=640ca93e-8e29-3b67-db81-00c6e918fd36]
vault_identity_group.hr_admin: Refreshing state... [id=a907b2a1-0f74-ee52-6451-fe282aa6b6bc]
vault

---

## 📜 Understanding Vault Policies

Vault policies define what authenticated entities can do. Let's examine the two policies we created:

### HR Basic Policy

This policy is for basic HR staff (like Alice). It allows:
- Reading credentials for the `hr-basic-reader` database role
- This role can only query `employee_basic_info` table

In [ ]:
basic_policy_path = lab_root / 'infrastructure' / 'vault' / 'policies' / 'hr-basic-policy.hcl'

print("📜 HR Basic Policy (hr-basic-policy.hcl):")
print("=" * 80)
with open(basic_policy_path, 'r') as f:
    policy_content = f.read()
    print(policy_content)
print("=" * 80)

print("\n💡 Key Points:")
print("  • Allows 'read' capability on 'database/creds/hr-basic-reader'")
print("  • This means Alice can request credentials for basic reader role")
print("  • The database role itself limits what queries can be executed")

### HR Admin Policy

This policy is for HR administrators (like Bob). It allows:
- Reading credentials for BOTH `hr-basic-reader` AND `hr-admin-reader` roles
- The admin role can query both `employee_basic_info` and `employee_salary_info` tables

In [ ]:
admin_policy_path = lab_root / 'infrastructure' / 'vault' / 'policies' / 'hr-admin-policy.hcl'

print("📜 HR Admin Policy (hr-admin-policy.hcl):")
print("=" * 80)
with open(admin_policy_path, 'r') as f:
    policy_content = f.read()
    print(policy_content)
print("=" * 80)

print("\n💡 Key Points:")
print("  • Allows 'read' on BOTH credential paths")
print("  • Bob can request credentials for both basic and admin roles")
print("  • This gives Bob access to all employee data")

### 🌐 View Policies in Vault UI

You can also view and explore these policies in the Vault web interface:

1. **Open Vault UI**: [http://localhost:8200/ui/vault/policies/acl](http://localhost:8200/ui/vault/policies/acl)
2. **Login Method**: Select "Token"
3. **Token**: Enter `root` (this is the root token)
4. **Click Sign In**

In the UI, you can:
- View all policies (hr-basic-policy, hr-admin-policy, default)
- See the policy syntax with syntax highlighting
- Explore other Vault configurations (auth methods, secrets engines, etc.)

💡 **Tip**: The Vault UI is a great way to visualize and understand your Vault configuration!

### Policy Comparison

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    'User': ['Alice', 'Bob'],
    'Group': ['hr-basic', 'hr-admin'],
    'Policy': ['hr-basic-policy', 'hr-basic-policy + hr-admin-policy'],
    'Can Access Basic Info': ['✅ Yes', '✅ Yes'],
    'Can Access Salary Info': ['❌ No', '✅ Yes'],
    'Database Roles': ['hr-basic-reader', 'hr-basic-reader + hr-admin-reader']
}

df = pd.DataFrame(comparison_data)
print("\n📊 Policy Comparison:")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)

print("\n🎯 This is Policy-Based Access Control (PBAC) in action!")
print("   • Policies are attached to user groups")
print("   • Groups come from IBM Verify")
print("   • Vault enforces policies before granting access")

---

## 🔗 External Group Mapping

External group mapping connects IBM Verify groups to Vault policies:

```
IBM Verify Group → Vault External Group → Vault Policy
```

### How It Works:

1. **User authenticates** with IBM Verify
2. **JWT token includes** user's groups (e.g., "hr-basic")
3. **Vault receives** JWT and validates it
4. **Vault maps** "hr-basic" group to "hr-basic-policy"
5. **Policy determines** what the user can access

### Configuration:

The configuration script created:

```bash
# External group for hr-basic
vault write identity/group name="hr-basic-external" \
  type="external" \
  policies="hr-basic-policy"

# External group for hr-admin  
vault write identity/group name="hr-admin-external" \
  type="external" \
  policies="hr-basic-policy,hr-admin-policy"
```

### Group Aliases:

Group aliases link IBM Verify group names to Vault external groups:

```bash
# Map IBM Verify "hr-basic" to Vault external group
vault write identity/group-alias name="hr-basic" \
  mount_accessor=<jwt_accessor> \
  canonical_id=<hr-basic-external-id>
```

### 🔗 External Group Mapping Summary

| IBM Verify Group | Vault External Group | Vault Policies |
|------------------|---------------------|----------------|
| `hr-basic` | `hr-basic-group` | `hr-basic-policy` |
| `hr-admin` | `hr-admin-group` | `hr-basic-policy`<br>`hr-admin-policy` |

### 💡 Benefits

1. **Centralized identity management** in IBM Verify
2. **Automatic policy assignment** based on groups
3. **No need to manage users directly** in Vault
4. **Easy to add/remove users** by changing groups in IBM Verify

### 🌐 View Groups in Vault UI

You can explore the identity groups and their mappings in the Vault web interface:

1. **Open Vault UI**: [http://localhost:8200/ui/vault/access/identity/groups](http://localhost:8200/ui/vault/access/identity/groups)
2. **Login**: Use Token method with `root` as the root token (if not already logged in)
3. **Explore Groups**: Click on each group to see:
   - Group name and type (external)
   - Assigned policies
   - Group aliases (mappings to IBM Verify groups)
   - Members (when users authenticate)

**Groups to explore:**
- `hr-basic-group` → Policies: hr-basic-policy (Alice's group - limited access)
- `hr-admin-group` → Policies: hr-basic-policy, hr-admin-policy (Bob's group - full access)

💡 **Tip**: The Identity section in Vault UI helps you visualize how external identities map to internal policies!

---

## 🗄️ Database Secrets Engine

The Database Secrets Engine generates dynamic database credentials. Here's how it works:

### Configuration:

1. **Database Connection** - Vault connects to PostgreSQL
2. **Database Roles** - Define SQL statements for creating users
3. **Credential Generation** - Vault creates temporary database users
4. **Automatic Revocation** - Vault deletes users when lease expires

### Database Roles Created:

#### hr-basic-reader
```sql
CREATE ROLE "{{name}}" WITH LOGIN PASSWORD '{{password}}' VALID UNTIL '{{expiration}}';
GRANT SELECT ON employee_basic_info TO "{{name}}";
```

#### hr-admin-reader
```sql
CREATE ROLE "{{name}}" WITH LOGIN PASSWORD '{{password}}' VALID UNTIL '{{expiration}}';
GRANT SELECT ON employee_basic_info TO "{{name}}";
GRANT SELECT ON employee_salary_info TO "{{name}}";
```

### 🗄️ Database Secrets Engine Configuration

| Role Name | Tables Accessible | Lease Duration |
|-----------|------------------|----------------|
| `hr-basic-reader` | `employee_basic_info` | 1 hour |
| `hr-admin-reader` | `employee_basic_info`<br>`employee_salary_info` | 1 hour |

### 💡 Dynamic Credentials Benefits

1. **No long-lived credentials** in code or config
2. **Credentials automatically expire** (1 hour TTL)
3. **Each request gets unique credentials**
4. **Vault automatically revokes** expired credentials
5. **Complete audit trail** of credential generation

### 🔒 Security Impact

- Eliminates credential sprawl
- Reduces blast radius of compromised credentials
- Automatic rotation without manual intervention
- Credentials tied to specific user identity

### 🌐 View Database Secrets Engine in Vault UI

You can explore the database secrets engine configuration in the Vault web interface:

1. **Open Vault UI**: [http://localhost:8200/ui/vault/secrets/database/overview](http://localhost:8200/ui/vault/secrets/database/overview)
2. **Login**: Use Token method with `root` as the root token (if not already logged in)
3. **Explore Configuration**:
   - **Connections** tab: View PostgreSQL connection details
   - **Roles** tab: See `hr-basic-reader` and `hr-admin-reader` roles
   - Click on each role to view:
     - Creation statements (SQL)
     - Default TTL and Max TTL
     - Database connection used

4. **Test Credential Generation** (optional):
   - Click on a role
   - Click "Generate credentials"
   - See temporary username and password created
   - Note the lease duration

💡 **Tip**: The Database Secrets Engine UI shows you exactly how Vault generates and manages dynamic credentials!

---

## ✅ Configuration Complete!

Congratulations! You have successfully configured HashiCorp Vault with:

- ✅ JWT Authentication Method
- ✅ Database Secrets Engine
- ✅ Two Vault Policies (hr-basic, hr-admin)
- ✅ External Group Mapping to IBM Verify
- ✅ Two Database Roles (hr-basic-reader, hr-admin-reader)

### 📝 What We Learned:

1. **Vault Policies** - Define fine-grained access control
2. **External Groups** - Map identity provider groups to policies
3. **Database Secrets Engine** - Generate dynamic database credentials
4. **Policy-Based Access Control** - Enforce permissions based on user groups

### 🎯 Next Steps:

Continue to **Part 3: Security Flow** (`03-security-flow.ipynb`) where you will:

- Test OAuth 2.0 token exchange
- Authenticate with Vault using JWT
- Generate dynamic database credentials
- Understand the complete security flow

---

**Ready to continue?** Open `03-security-flow.ipynb` to proceed!